# 🧬 MolGen v2 — SMILES Transformer + RL Fine-tuning

**Target:** 85% validity, 75% novelty

**Architecture:** SMILES Tokenizer → Transformer Decoder → RL fine-tuning (QED reward)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Mount Google Drive (for checkpoint saving)

**Estimated time on T4 GPU:** ~7-8 hours total (run across 2-3 Colab sessions)

**Session plan:**
- Session 1 (~90 mins): Epochs 1–100 saved to Drive
- Session 2 (~90 mins): Epochs 101–200 resumed from checkpoint
- Session 3 (~90 mins): Epochs 201–300 + RL fine-tuning

**Checkpoints auto-save every epoch to Google Drive — never lose progress!**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install rdkit torch tqdm pandas numpy -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, AllChem
from tqdm import tqdm
import math, os, json, re
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

✅ Using device: cuda
   GPU: Tesla T4


In [ ]:
# ── Cell 2: Mount Google Drive (for checkpoint saving) ────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/molgen_v2'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Checkpoints will be saved to: {SAVE_DIR}')

Mounted at /content/drive
✅ Checkpoints will be saved to: /content/drive/MyDrive/molgen_v2


In [ ]:
# ── Cell 3: Load dataset ───────────────────────────────────────────────────────
import os

if not os.path.exists('SMILES_Big_Data_Set.csv'):
    from google.colab import files
    print('Upload SMILES_Big_Data_Set.csv from your computer...')
    files.upload()

df = pd.read_csv('SMILES_Big_Data_Set.csv')
all_smiles = df['SMILES'].dropna().unique().tolist()
print(f'Total SMILES loaded: {len(all_smiles)}')

# Filter: keep only valid, reasonably sized molecules
valid_smiles = []
for s in tqdm(all_smiles, desc='Validating SMILES'):
    mol = Chem.MolFromSmiles(s)
    if mol and 5 <= len(s) <= 120:   # filter very short/long molecules
        valid_smiles.append(s)

print(f'✅ Valid SMILES after filtering: {len(valid_smiles)}')

Upload SMILES_Big_Data_Set.csv from your computer...


Saving SMILES_Big_Data_Set.csv to SMILES_Big_Data_Set.csv
Total SMILES loaded: 15872


Validating SMILES: 100%|██████████| 15872/15872 [00:02<00:00, 6233.05it/s]

✅ Valid SMILES after filtering: 15837


In [ ]:
# ── Cell 4: SMILES Tokenizer ───────────────────────────────────────────────────
# Tokenizes SMILES character by character with special tokens

class SMILESTokenizer:
    """
    Tokenizes SMILES strings using regex to correctly handle
    multi-character tokens like Cl, Br, [NH], etc.
    """
    PATTERN = r'(\[[^\]]+\]|Br|Cl|Si|Se|se|As|te|Te|@@|@|%\d{2}|[A-Za-z]|[0-9]|[\+\-\.\#\=\/\\\(\)\[\]]|.)'

    PAD   = '<PAD>'
    SOS   = '<SOS>'   # start of sequence
    EOS   = '<EOS>'   # end of sequence
    UNK   = '<UNK>'

    def __init__(self):
        self.token2idx = {}
        self.idx2token = {}
        self.built     = False

    def tokenize(self, smiles: str):
        return re.findall(self.PATTERN, smiles)

    def build_vocab(self, smiles_list):
        counter = Counter()
        for s in smiles_list:
            counter.update(self.tokenize(s))
        special = [self.PAD, self.SOS, self.EOS, self.UNK]
        vocab   = special + sorted(counter.keys())
        self.token2idx = {t: i for i, t in enumerate(vocab)}
        self.idx2token = {i: t for t, i in self.token2idx.items()}
        self.built     = True
        print(f'✅ Vocabulary size: {len(self.token2idx)}')

    def encode(self, smiles: str, max_len: int = 122):
        tokens = [self.SOS] + self.tokenize(smiles) + [self.EOS]
        ids    = [self.token2idx.get(t, self.token2idx[self.UNK]) for t in tokens]
        # Pad or truncate
        ids = ids[:max_len]
        ids += [self.token2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        tokens = [self.idx2token.get(i, self.UNK) for i in ids]
        smiles = ''
        for t in tokens:
            if t in (self.SOS, self.PAD):
                continue
            if t == self.EOS:
                break
            smiles += t
        return smiles

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({'token2idx': self.token2idx}, f)

    def load(self, path):
        with open(path) as f:
            d = json.load(f)
        self.token2idx = d['token2idx']
        self.idx2token = {int(i): t for t, i in self.token2idx.items()}
        self.built     = True


tokenizer = SMILESTokenizer()
tokenizer.build_vocab(valid_smiles)
tokenizer.save(os.path.join(SAVE_DIR, 'tokenizer.json'))

VOCAB_SIZE = len(tokenizer.token2idx)
PAD_IDX    = tokenizer.token2idx[SMILESTokenizer.PAD]
SOS_IDX    = tokenizer.token2idx[SMILESTokenizer.SOS]
EOS_IDX    = tokenizer.token2idx[SMILESTokenizer.EOS]
MAX_LEN    = 122

print(f'PAD={PAD_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}')

✅ Vocabulary size: 45
PAD=0, SOS=1, EOS=2


In [ ]:
# ── Cell 5: Dataset ────────────────────────────────────────────────────────────

class SMILESDataset(Dataset):
    def __init__(self, smiles_list, tokenizer, max_len=122):
        self.data      = smiles_list
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = self.tokenizer.encode(self.data[idx], self.max_len)
        ids = torch.tensor(ids, dtype=torch.long)
        # Input: SOS + tokens, Target: tokens + EOS
        return ids[:-1], ids[1:]


# 90/10 train/val split
split      = int(0.9 * len(valid_smiles))
train_smi  = valid_smiles[:split]
val_smi    = valid_smiles[split:]

train_ds   = SMILESDataset(train_smi, tokenizer, MAX_LEN)
val_ds     = SMILESDataset(val_smi,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

✅ Train: 14253 | Val: 1584


In [ ]:
# ── Cell 6: Transformer Model ──────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 200):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class MolTransformer(nn.Module):
    """
    Autoregressive Transformer decoder for SMILES generation.
    Generates one token at a time conditioned on previous tokens.
    """
    def __init__(self, vocab_size, d_model=256, nhead=8,
                 num_layers=6, dim_ff=1024, dropout=0.1, max_len=122):
        super().__init__()
        self.d_model   = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc   = PositionalEncoding(d_model, dropout, max_len + 10)
        decoder_layer  = nn.TransformerDecoderLayer(
            d_model, nhead, dim_ff, dropout, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        self.output      = nn.Linear(d_model, vocab_size)

        # Memory (empty — pure decoder, no encoder)
        self.register_buffer('_mem_dummy', torch.zeros(1, 1, d_model))

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, std=0.02)
        nn.init.zeros_(self.output.bias)
        nn.init.normal_(self.output.weight, std=0.02)

    def forward(self, x):
        """
        x: (B, T) token ids
        returns: (B, T, vocab_size) logits
        """
        B, T   = x.shape
        emb    = self.pos_enc(self.embedding(x) * math.sqrt(self.d_model))
        # Causal mask: token i can only attend to tokens 0..i
        mask   = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        pad_m  = (x == PAD_IDX)                      # (B, T) padding mask
        mem    = self._mem_dummy.expand(B, -1, -1)    # dummy memory
        out    = self.transformer(emb, mem,
                                  tgt_mask=mask,
                                  tgt_key_padding_mask=pad_m)
        return self.output(out)                        # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, n: int = 1, max_len: int = 120,
             temperature: float = 1.0, top_k: int = 10):
        device_ = next(self.parameters()).device
        self.eval()
        ids  = torch.full((n, 1), SOS_IDX, dtype=torch.long, device=device_)
        done = torch.zeros(n, dtype=torch.bool, device=device_)

        UNK_IDX   = tokenizer.token2idx.get('<UNK>', -1)
        CLOSE_IDX = tokenizer.token2idx.get(')', -1)

    # Pre-compute digit token indices for ring tracking
        DIGIT_IDXS = {tokenizer.token2idx[d]: d
                  for d in '123456789' if d in tokenizer.token2idx}
    # Pre-compute %NN token indices
        PCT_IDXS = {tokenizer.token2idx[t]: t
                for t in tokenizer.token2idx if re.match(r'^%\d{2}$', t)}

    # Per-sequence structural state
        open_parens = [0] * n
        open_rings  = [set() for _ in range(n)]  # ring ids currently open

        for _ in range(max_len):
            logits = self.forward(ids)[:, -1, :] / temperature

        # Block forbidden tokens globally
            logits[:, PAD_IDX] = float('-inf')
            logits[:, SOS_IDX] = float('-inf')
            if UNK_IDX >= 0:
                logits[:, UNK_IDX] = float('-inf')

        # Per-sequence structural constraints
            for i in range(n):
                if done[i]:
                    continue
            # Block ) if no open paren
                if open_parens[i] == 0 and CLOSE_IDX >= 0:
                    logits[i, CLOSE_IDX] = float('-inf')
            # Block EOS if parens or rings still open
                if open_parens[i] > 0 or len(open_rings[i]) > 0:
                    logits[i, EOS_IDX] = float('-inf')
            # Block re-opening a ring digit that is already open
            # (would accidentally close it mid-sequence at wrong place)
                for didx, dtok in DIGIT_IDXS.items():
                    if len(open_rings[i]) >= 4 and dtok not in open_rings[i]:
                        logits[i, didx] = float('-inf')

        # Top-k sampling
            if top_k > 0:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits  = logits.masked_fill(logits < vals[:, -1:], float('-inf'))

            probs   = torch.softmax(logits, dim=-1)
            probs   = torch.nan_to_num(probs, nan=1.0/probs.size(-1))
            probs   = probs / probs.sum(dim=-1, keepdim=True)
            next_id = torch.multinomial(probs, 1)

        # Update structural state
            for i in range(n):
                if done[i]:
                    continue
                tok = tokenizer.idx2token.get(next_id[i].item(), '')
                if tok == '(':
                    open_parens[i] += 1
                elif tok == ')':
                    open_parens[i] = max(0, open_parens[i] - 1)
                elif tok in DIGIT_IDXS.values():
                    if tok in open_rings[i]:
                        open_rings[i].discard(tok)
                    else:
                        open_rings[i].add(tok)
                elif tok in PCT_IDXS.values():
                    if tok in open_rings[i]:
                        open_rings[i].discard(tok)
                    else:
                        open_rings[i].add(tok)

            ids  = torch.cat([ids, next_id], dim=1)
            done = done | (next_id.squeeze(1) == EOS_IDX)
            if done.all():
                break

        return [tokenizer.decode(ids[i].tolist()) for i in range(n)]


model = MolTransformer(
    vocab_size = VOCAB_SIZE,
    d_model    = 256,
    nhead      = 8,
    num_layers = 6,
    dim_ff     = 1024,
    dropout    = 0.1,
    max_len    = MAX_LEN
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ MolTransformer | Parameters: {total_params:,}')

✅ MolTransformer | Parameters: 6,343,725


In [ ]:
# ── Cell 7: Training helpers ───────────────────────────────────────────────────

def compute_validity(smiles_list):
    valid = [s for s in smiles_list if Chem.MolFromSmiles(s) is not None]
    return len(valid) / len(smiles_list) * 100 if smiles_list else 0, valid

def compute_metrics(smiles_list, train_set):
    if not smiles_list:
        return 0, 0, 0, 0
    valid_pct, valid = compute_validity(smiles_list)
    unique      = list(set(valid))
    uniqueness  = len(unique) / len(smiles_list) * 100
    novelty     = sum(1 for s in unique if s not in train_set) / len(unique) * 100 if unique else 0
    qeds        = [QED.qed(Chem.MolFromSmiles(s)) for s in unique if Chem.MolFromSmiles(s)]
    avg_qed     = np.mean(qeds) if qeds else 0
    return round(valid_pct,1), round(uniqueness,1), round(novelty,1), round(avg_qed,3)

train_set_lookup = set(valid_smiles)

def evaluate_model(model, n=200):
    samples = model.generate(n=n, temperature=1.0, top_k=10)
    v, u, nov, q = compute_metrics(samples, train_set_lookup)
    print(f'  Validity={v}%  Uniqueness={u}%  Novelty={nov}%  QED={q}')
    return v, u, nov, q

In [ ]:
# ── Cell 8: Phase 1 — Supervised Training ─────────────────────────────────────
# Train the Transformer to reconstruct training SMILES (teacher forcing)
# 300 epochs needed to reach ~75% validity before RL fine-tuning
# Expected time on T4 GPU: ~5-6 hours (run overnight or across 2 sessions)
# Checkpoints saved every epoch — safe to disconnect and resume anytime

EPOCHS_PHASE1  = 300   # 300 epochs needed for 85% validity target
best_val_loss  = float('inf')
loss_history   = []

optimizer  = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1)
criterion  = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ── Resume from checkpoint if Colab disconnected ──
ckpt_path   = os.path.join(SAVE_DIR, 'phase1_best.pth')
latest_path = os.path.join(SAVE_DIR, 'phase1_latest.pth')  # always saved, not just best
start_epoch = 1

if os.path.exists(latest_path):
    ckpt = torch.load(latest_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    loss_history  = ckpt.get('loss_history', [])
    print(f'✅ Resumed from epoch {ckpt["epoch"]}/{EPOCHS_PHASE1} (best_val={best_val_loss:.4f})')
    print(f'   Continuing from epoch {start_epoch}...')
else:
    print('🚀 Starting Phase 1 training from scratch...')

print()
print(f'Training for {EPOCHS_PHASE1} epochs on {len(train_ds)} molecules...')
print(f'Estimated time on T4 GPU: ~5-6 hours')
print(f'💡 Tip: Run overnight or use 2 Colab sessions — checkpoints auto-save')
print()

# Expected validity by epoch:
# Epoch  50 → ~40-50%
# Epoch 100 → ~55-65%
# Epoch 200 → ~68-75%
# Epoch 300 → ~75-80%  ← ready for RL fine-tuning

for epoch in range(start_epoch, EPOCHS_PHASE1 + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        logits   = model(src)                          # (B, T, V)
        loss     = criterion(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits   = model(src)
            val_loss += criterion(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1)).item()

    scheduler.step()
    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    loss_history.append((avg_train, avg_val))

    # ── Save best checkpoint ──
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'model':      model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'scheduler':  scheduler.state_dict(),
            'epoch':      epoch,
            'val_loss':   avg_val,
            'best_val_loss': best_val_loss,
            'loss_history':  loss_history,
            'vocab_size': VOCAB_SIZE,
        }, ckpt_path)

    # ── Always save latest (so we can resume after disconnect) ──
    torch.save({
        'model':         model.state_dict(),
        'optimizer':     optimizer.state_dict(),
        'scheduler':     scheduler.state_dict(),
        'epoch':         epoch,
        'best_val_loss': best_val_loss,
        'loss_history':  loss_history,
    }, latest_path)

    if epoch % 10 == 0:
        lr  = scheduler.get_last_lr()[0]
        pct = int(epoch / EPOCHS_PHASE1 * 100)
        print(f'Epoch {epoch:3d}/{EPOCHS_PHASE1} ({pct:2d}%) | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {lr:.2e}')

    # ── Evaluate every 50 epochs ──
    if epoch % 50 == 0:
        print(f'  → Evaluating (100 samples)...')
        model.eval()
        evaluate_model(model, n=100)
        print()

print(f'\n✅ Phase 1 complete. Best val loss: {best_val_loss:.4f}')
print('Now run Cell 9 to evaluate, then Cell 10 for RL fine-tuning.')

✅ Resumed from epoch 300/300 (best_val=0.8619)
   Continuing from epoch 301...

Training for 300 epochs on 14253 molecules...
Estimated time on T4 GPU: ~5-6 hours
💡 Tip: Run overnight or use 2 Colab sessions — checkpoints auto-save


✅ Phase 1 complete. Best val loss: 0.8619
Now run Cell 9 to evaluate, then Cell 10 for RL fine-tuning.


In [ ]:
# ── Cell 9: Phase 1 Evaluation ────────────────────────────────────────────────
# Load best Phase 1 checkpoint and evaluate

ckpt = torch.load(os.path.join(SAVE_DIR, 'phase1_best.pth'), map_location=device)
model.load_state_dict(ckpt['model'])
print(f'✅ Loaded best Phase 1 model (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')
print()
print('Evaluating Phase 1 model (200 samples)...')
v, u, nov, q = evaluate_model(model, n=200)

print()
print('='*45)
print('  PHASE 1 RESULTS')
print('='*45)
print(f'  Validity   : {v}%')
print(f'  Uniqueness : {u}%')
print(f'  Novelty    : {nov}%')
print(f'  Avg QED    : {q}')
print('='*45)
print()
print('Expected at this stage: Validity ~65-75%, Novelty ~50-65%')
print('Now proceeding to Phase 2: RL fine-tuning to push higher...')

✅ Loaded best Phase 1 model (epoch 33, val_loss=0.8619)

Evaluating Phase 1 model (200 samples)...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
[09:30:54] Can't kekulize mol.  Unkekulized atoms: 2 4 5 6 7 8 9 10 11 12 13
[09:30:54] Can't kekulize mol.  Unkekulized atoms: 2 5 6 8
[09:30:54] Explicit valence for atom # 13 C, 6, is greater than permitted
[09:30:54] Can't kekulize mol.  Unkekulized atoms: 2 4 6 9 11
[09:30:54] SMILES Parse Error: ring closure 1 duplicates bond between atom 50 and atom 51 for input: 'CCCC(=O)OCC(=O)C1(OC(=O)OCC)CC3CCC4=CC(=O)CC=CC4(C)C3(F)C(O)CCC21C(O)CC(O)C(O)CC(=O)CC(C)OC12C1C'
[09:30:54] SMILES Parse Error: syntax error while parsing: CC1OC(OC2C(O)CC(OC3C(O)C(OC4CCCC4C3CCC5(C)C(C5=CC(=O)OC5)C(O)CCC4)OC(C)CC4)OC3C2C)OC2C(C)C(OCOC(O)C(O)C(=O)C(O)C(O)C(OC(
[09:30:54] SMILES Parse Error: check for mistakes around position 120:
[09:30:54] (O)C(=O)C(O)C(O)C(OC(
[09:30:54] ~~

  Validity=82.5%  Uniqueness=82.5%  Novelty=68.5%  QED=0.607

  PHASE 1 RESULTS
  Validity   : 82.5%
  Uniqueness : 82.5%
  Novelty    : 68.5%
  Avg QED    : 0.607

Expected at this stage: Validity ~65-75%, Novelty ~50-65%
Now proceeding to Phase 2: RL fine-tuning to push higher...


In [ ]:
import gc
import os
import numpy as np
from collections import Counter, deque

import torch
import torch.optim as optim
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

torch.cuda.empty_cache()
gc.collect()

# ── Diversity tracking ────────────────────────────────────────────────────────
recent_molecules = deque(maxlen=200)
recent_counter   = Counter()


def compute_reward(smiles: str, train_set: set) -> float:
    # Hard penalty for empty or garbage strings
    if not smiles or len(smiles) < 3:
        return -2.0

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        if '<UNK>' in smiles:
            return -2.0
        if smiles.count('(') != smiles.count(')'):
            return -1.5
        if smiles[0] in (')', '=', '-', '#'):
            return -1.5
        return -1.0

    if mol.GetNumAtoms() < 3:
        return -0.5

    qed_score = QED.qed(mol)
    mw  = Descriptors.MolWt(mol)
    lp  = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    lipinski = 1.0 if (mw <= 500 and lp <= 5 and hbd <= 5 and hba <= 10) else 0.5
    novelty  = 0.3 if smiles not in train_set else 0.0

    recent_count      = recent_counter.get(smiles, 0)
    diversity_penalty = 0.3 * min(recent_count, 5) / 5

    reward = qed_score * lipinski + novelty - diversity_penalty

    if len(recent_molecules) == recent_molecules.maxlen:
        evicted = recent_molecules[0]
        recent_counter[evicted] -= 1
        if recent_counter[evicted] == 0:
            del recent_counter[evicted]

    recent_molecules.append(smiles)
    recent_counter[smiles] = recent_counter.get(smiles, 0) + 1

    return reward


def rl_step(model, optimizer, n_samples=8, temperature=1.1):
    model.train()
    torch.cuda.empty_cache()
    B         = n_samples
    ids       = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
    log_probs = []
    done_mask = torch.zeros(B, dtype=torch.bool, device=device)

    # Pre-compute token indices
    UNK_IDX   = tokenizer.token2idx.get('<UNK>', -1)
    CLOSE_IDX = tokenizer.token2idx.get(')', -1)
    DIGIT_IDXS = {tokenizer.token2idx[d]: d
                  for d in '123456789' if d in tokenizer.token2idx}
    PCT_IDXS = {tokenizer.token2idx[t]: t
                for t in tokenizer.token2idx if re.match(r'^%\d{2}$', t)}

    # Per-sequence structural state
    open_parens = [0] * B
    open_rings  = [set() for _ in range(B)]

    for step in range(MAX_LEN - 1):
        with torch.amp.autocast('cuda'):
            logits = model(ids)[:, -1, :] / temperature

        logits = logits.float()

        # Block forbidden tokens globally
        logits[:, PAD_IDX] = float('-inf')
        logits[:, SOS_IDX] = float('-inf')
        if UNK_IDX >= 0:
            logits[:, UNK_IDX] = float('-inf')

        # Per-sequence structural constraints
        for i in range(B):
            if done_mask[i]:
                continue
            if open_parens[i] == 0 and CLOSE_IDX >= 0:
                logits[i, CLOSE_IDX] = float('-inf')
            if open_parens[i] > 0 or len(open_rings[i]) > 0:
                logits[i, EOS_IDX] = float('-inf')
            for didx, dtok in DIGIT_IDXS.items():
                if len(open_rings[i]) >= 4 and dtok not in open_rings[i]:
                    logits[i, didx] = float('-inf')

        probs   = torch.softmax(logits, dim=-1)
        dist    = torch.distributions.Categorical(probs)
        next_id = dist.sample()
        lp      = dist.log_prob(next_id)
        lp      = lp.masked_fill(done_mask, 0.0)
        log_probs.append(lp)

        # Update structural state
        for i in range(B):
            if done_mask[i]:
                continue
            tok = tokenizer.idx2token.get(next_id[i].item(), '')
            if tok == '(':
                open_parens[i] += 1
            elif tok == ')':
                open_parens[i] = max(0, open_parens[i] - 1)
            elif tok in DIGIT_IDXS.values():
                if tok in open_rings[i]:
                    open_rings[i].discard(tok)
                else:
                    open_rings[i].add(tok)
            elif tok in PCT_IDXS.values():
                if tok in open_rings[i]:
                    open_rings[i].discard(tok)
                else:
                    open_rings[i].add(tok)

        ids       = torch.cat([ids, next_id.unsqueeze(1)], dim=1)
        done_mask = done_mask | (next_id == EOS_IDX)
        if done_mask.all():
            break

    smiles_list = [tokenizer.decode(ids[i].tolist()) for i in range(B)]
    rewards     = torch.tensor(
        [compute_reward(s, train_set_lookup) for s in smiles_list],
        dtype=torch.float32, device=device
    )
    advantages   = rewards - rewards.mean()
    log_probs_t  = torch.stack(log_probs, dim=1)
    seq_log_prob = log_probs_t.sum(dim=1)
    loss         = -(advantages.detach() * seq_log_prob).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    optimizer.step()

    avg_reward = rewards.mean().item()
    del ids, log_probs, log_probs_t, rewards, advantages, seq_log_prob
    torch.cuda.empty_cache()
    return loss.item(), avg_reward, smiles_list


# ── Settings ──────────────────────────────────────────────────────────────────
EPOCHS_PHASE2  = 60
RL_BATCH_SIZE  = 8
RL_STEPS       = 40
rl_optimizer   = optim.Adam(model.parameters(), lr=3e-6)
rl_ckpt_path   = os.path.join(SAVE_DIR, 'phase2_best.pth')
rl_latest_path = os.path.join(SAVE_DIR, 'phase2_latest.pth')
best_validity  = 0.0

scaler = torch.amp.GradScaler('cuda')

# ── Load checkpoint ───────────────────────────────────────────────────────────
if os.path.exists(rl_ckpt_path):
    ckpt = torch.load(rl_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    best_validity = ckpt.get('validity', 0)
    rl_start      = ckpt['epoch'] + 1
    print(f'🔄 Rolled back to epoch {ckpt["epoch"]} | validity={ckpt["validity"]}% novelty={ckpt["novelty"]}%')
elif os.path.exists(rl_latest_path):
    ckpt = torch.load(rl_latest_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    rl_start      = ckpt['epoch'] + 1
    best_validity = ckpt.get('best_validity', 0)
    print(f'✅ Resumed RL from epoch {ckpt["epoch"]}/{EPOCHS_PHASE2}')
else:
    rl_start = 1
    print('🚀 Starting Phase 2 RL fine-tuning...')

print(f'Batch={RL_BATCH_SIZE}, Steps={RL_STEPS}, Epochs={EPOCHS_PHASE2}')
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB')
print()

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(rl_start, EPOCHS_PHASE2 + 1):
    epoch_losses, epoch_rewards = [], []

    for _ in range(RL_STEPS):
        loss, avg_reward, _ = rl_step(model, rl_optimizer,
                                      n_samples=RL_BATCH_SIZE,
                                      temperature=1.1)
        epoch_losses.append(loss)
        epoch_rewards.append(avg_reward)

    avg_loss   = np.mean(epoch_losses)
    avg_reward = np.mean(epoch_rewards)

    torch.save({
        'model':         model.state_dict(),
        'epoch':         epoch,
        'best_validity': best_validity,
    }, rl_latest_path)

    if epoch % 5 == 0:
        print(f'RL Epoch {epoch:2d}/{EPOCHS_PHASE2} | Loss: {avg_loss:.4f} | Reward: {avg_reward:.4f}')
        model.eval()
        v, u, nov, q = evaluate_model(model, n=100)
        if v > best_validity:
            best_validity = v
            torch.save({
                'model':    model.state_dict(),
                'epoch':    epoch,
                'validity': v,
                'novelty':  nov,
                'qed':      q,
            }, rl_ckpt_path)
            print(f'  ✅ Best model saved (validity={v}%)')
        print()

print(f'✅ Phase 2 complete. Best validity: {best_validity:.1f}%')

🔄 Rolled back to epoch 25 | validity=100.0% novelty=97.0%
Batch=8, Steps=40, Epochs=60
GPU free: 14.25 GB



[09:30:58] Can't kekulize mol.  Unkekulized atoms: 23 25 26 27 28
[09:31:00] Can't kekulize mol.  Unkekulized atoms: 13 14 15 16 17
[09:31:00] Explicit valence for atom # 8 F, 2, is greater than permitted
[09:31:05] SMILES Parse Error: unclosed ring for input: 'Cc1ccc(Cl)c([N+](O)=O)c1Occcc1OCCClCClCCClC(F)ClCClCClCClCClCC(C)CCCClCClCCCClCCC5C4CCCC3CC43CCC1CCCCC21(O)cc1FCC(=O)OCC(C)c1(CBr)N(C)C1Cl'
[09:31:14] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[09:31:19] Explicit valence for atom # 16 Cl, 2, is greater than permitted
[09:31:22] SMILES Parse Error: ring closure 1 duplicates bond between atom 65 and atom 66 for input: 'OCc1ccccc1OC(c[nH]c2cccc1)c1ccccc1F6ClFOC(Cl)CC5CCC2CC(O)CO4C(F)C(CO)CC1COCC(C)C(C)O1CCC1C52CCCC1CCC1C1C1CCCC1CC1COC1C1CC1CC1C'
[09:31:26] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 10 11 12 13 14 15 19 20 21 22 23 24
[09:31:26] Explicit valence for atom # 18 F, 2, is greater than permitted
[09:31:28] Can't kekulize mol.  Unkekulized atoms: 2 3 4

RL Epoch 30/60 | Loss: -2.4916 | Reward: 0.8627
  Validity=100.0%  Uniqueness=100.0%  Novelty=95.0%  QED=0.744



[09:36:58] Can't kekulize mol.  Unkekulized atoms: 1 2 3 17 18
[09:37:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 13 18
[09:37:06] non-ring atom 8 marked aromatic
[09:37:07] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20 21 22
[09:37:09] Explicit valence for atom # 7 N, 5, is greater than permitted
[09:37:10] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 6
[09:37:12] Can't kekulize mol.  Unkekulized atoms: 2 3 4 6 7 8 13 14 15 16 17
[09:37:12] Can't kekulize mol.  Unkekulized atoms: 10 11 12 13 15
[09:37:12] Can't kekulize mol.  Unkekulized atoms: 4 5 7 8 28
[09:37:19] Explicit valence for atom # 3 O, 3, is greater than permitted
[09:37:20] Explicit valence for atom # 7 O, 4, is greater than permitted
[09:37:20] Can't kekulize mol.  Unkekulized atoms: 0 1 3 11 17
[09:37:20] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 16 17
[09:37:27] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20
[09:37:27] SMILES Parse Error: unclosed ring for input: 'CC(C)(C)NCC1CCC(

RL Epoch 35/60 | Loss: -3.9114 | Reward: 0.8635


...

[09:54:22] Explicit valence for atom # 10 Cl, 2, is greater than permitted
[09:54:26] Explicit valence for atom # 3 F, 2, is greater than permitted
[09:54:28] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5
[09:54:32] Can't kekulize mol.  Unkekulized atoms: 4 5 7
[09:54:32] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 12
[09:54:32] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 17
[09:54:33] Can't kekulize mol.  Unkekulized atoms: 7 8 9
[09:54:33] Explicit valence for atom # 15 S, 7, is greater than permitted
[09:54:37] SMILES Parse Error: unclosed ring for input: 'Cc1cccc(Cc2ccccc2)c1NC(=O)c1cccc(N2CCOCC)c1OC(C)CCC1CCCCCc1ccccc1Cl6c2CNCC1C1=O[SiH]1NOCCCC1C-c11=OCC1=CCC1CCCN1OCC21NC1C1BrPC1'
[09:54:37] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 9 10 11 12 15
[09:54:43] Explicit valence for atom # 29 F, 2, is greater than permitted
[09:54:47] non-ring atom 7 marked aromatic
[09:54:47] Can't kekulize mol.  Unkekulized atoms: 12 13 14 15 16 22 23
[09:54:53] Can't kekulize m

RL Epoch 50/60 | Loss: -3.5190 | Reward: 0.9105


[10:00:02] SMILES Parse Error: duplicated ring closure 5 bonds atom 46 to itself for input: 'Cc1cccc(NC2=CCN(C)C)c1C(=O)OCCCCCCc1ccccc1ClCFC(=O)OCCC(C)CClOCCC4CC2545'
[10:00:02] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 9


  Validity=98.0%  Uniqueness=97.0%  Novelty=91.8%  QED=0.757



[10:00:04] Explicit valence for atom # 7 F, 2, is greater than permitted
[10:00:06] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7
[10:00:07] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 9 10 11 12 13
[10:00:10] Explicit valence for atom # 10 O, 4, is greater than permitted
[10:00:13] SMILES Parse Error: unclosed ring for input: 'CN(c1[nH]cnc2nccc1-3)C(=O)NC(Cc1ccccc1)NC(C)=O=OCl(F)C(CC)(C)CC(C)CCBrC4C3C(=O)c1FOCCC(=O)Ccccc1=Cc1CN2cccccccccc1C2CC1CCc1C1C'
[10:00:13] Explicit valence for atom # 16 C, 6, is greater than permitted
[10:00:18] Can't kekulize mol.  Unkekulized atoms: 3 4 5 8 14
[10:00:24] Explicit valence for atom # 43 F, 2, is greater than permitted
[10:00:24] Can't kekulize mol.  Unkekulized atoms: 5 6 7 9 10 11 14
[10:00:26] Can't kekulize mol.  Unkekulized atoms: 13 14 15 16 17 20 21
[10:00:30] Explicit valence for atom # 17 Cl, 2, is greater than permitted
[10:00:32] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 18
[10:00:33] Can't kekulize mol.  Unkekulized ato

In [ ]:
# ── Cell 11: Final Evaluation ─────────────────────────────────────────────────
# Load best model (Phase 2 if available, else Phase 1)
if os.path.exists(rl_ckpt_path):
    ckpt = torch.load(rl_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f'✅ Loaded best RL model (epoch {ckpt["epoch"]})')
else:
    ckpt = torch.load(os.path.join(SAVE_DIR, 'phase1_best.pth'), map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f'✅ Loaded best Phase 1 model')

model.eval()
print('\nFinal evaluation (500 samples)...')
samples = model.generate(n=500, temperature=1.0, top_k=10)
v, u, nov, q = compute_metrics(samples, train_set_lookup)
print()
print('='*45)
print('  FINAL RESULTS (MolGen v2)')
print('='*45)
print(f'  Validity    : {v}%   (target: 85%)')
print(f'  Uniqueness  : {u}%')
print(f'  Novelty     : {nov}%   (target: 75%)')
print(f'  Avg QED     : {q}')
print('='*45)

In [ ]:
# ── Cell 12: Export model.pth for app.py ──────────────────────────────────────
# Saves model weights + tokenizer bundled together so app.py can't mismatch them

torch.save({
    'model':      model.state_dict(),
    'token2idx':  tokenizer.token2idx,
    'vocab_size': VOCAB_SIZE,
    'pad_idx':    PAD_IDX,
    'sos_idx':    SOS_IDX,
    'eos_idx':    EOS_IDX,
}, 'model_v2.pth')

import shutil
shutil.copy('model_v2.pth', os.path.join(SAVE_DIR, 'model_v2.pth'))

from google.colab import files
files.download('model_v2.pth')

print('✅ Saved model_v2.pth with embedded tokenizer')
print('Place this file in your Flask app folder alongside app.py')

In [ ]:
# ── Cell 13: Updated app.py snippet ───────────────────────────────────────────
# Copy this into your app.py to use the new Transformer model

print('''
# ── Paste this into app.py to use MolGen v2 ──────────────────────────────────

import re, json, math

# 1. Load tokenizer
with open("tokenizer.json") as f:
    token2idx = json.load(f)["token2idx"]
idx2token = {int(i): t for t, i in token2idx.items()}
VOCAB_SIZE = len(token2idx)
PAD_IDX = token2idx["<PAD>"]
SOS_IDX = token2idx["<SOS>"]
EOS_IDX = token2idx["<EOS>"]
MAX_LEN = 122

# 2. Load model
model = MolTransformer(VOCAB_SIZE).to(device)
model.load_state_dict(torch.load("model_v2.pth", map_location=device))
model.eval()

# 3. Generate in /predict endpoint
gen_smiles_list = model.generate(n=5, temperature=1.0, top_k=10)
# Filter for valid ones
valid_gen = [s for s in gen_smiles_list if Chem.MolFromSmiles(s)]
gen_smiles = valid_gen[0] if valid_gen else None
''')

In [ ]:
import os

# Search for the checkpoint files
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'phase2' in f or 'model_v2' in f or 'molgen' in f.lower():
            print(os.path.join(root, f))

In [ ]:
import os
for root, dirs, files_list in os.walk('/content'):
    for f in files_list:
        if 'phase2' in f or 'model_v2' in f:
            print(os.path.join(root, f))

In [ ]:
import os

# Search entire Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.pth'):
            print(os.path.join(root, f))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')